# HAC Steel OCBF — Lateral System Behavior, Rocking Overturning & Fuse Force

**Longitudinal load path + transverse rocking restoring & single-rod fuse sizing — ASCE 7-22 (Ch. 12 & 15), AISC 360-22 / 341-22, 2025 CBC**

| field | value |
|---|---|
| Project | OAI Richmond Robotic Lab — Phase 2 |
| Job No. | DG26.0160.00 |
| Location | 1411 Harbour Way S., Richmond, CA |
| Date | 2026-08-02 |
| Subject | HAC lateral system — longitudinal behavior · transverse rocking restoring · single-rod fuse force |
| Prepared | Jeffrey Dai (Structural Intern) |
| Checked | S. Aher (pending) |

**Basis.** 2025 CBC / ASCE 7-22 (Ch. 12 & 15); AISC 360-22 & 341-22. The permit
design stays code-based (ELF, Steel OCBF, `R = 3.25`, `Cs = 0.402`, RC II /
`Ie = 1.0`); this sheet supports the beyond-code PBD rocking-fuse options study.
Units are carried in kip–in–ksi via `forallpeople`; seismic coefficients are in
units of `g`.

**Purpose.** Companion to *OCBF_BaseShear_CapacityReconciliation*. Two questions
worked here: (1) **how the combined longitudinal system** — one central chevron-
braced bay plus welded moment frames in the other bays — distributes seismic
force; and (2) the **transverse rocking overturning**, specifically which base
width sets the self-weight restoring, then the resulting single-rod fuse force in
each direction.

> **PRELIMINARY — PBD OPTIONS STUDY — NOT FOR PERMIT / CONSTRUCTION**

> **Inputs locked in this sheet.** Whole-HAC RISA dead `W ≈ 230 kip` (bulk in the
> interior five 9′-8″ bays). Transverse base geometry refined: **vertical columns
> 8 ft apart, diagonal brace feet 14.5 ft apart** (feet 3′-3″ outboard of each
> column, S3.04/1). Both the vertical columns and the diagonal braces are
> telescoped with a single ductile threaded-rod fuse. Governing base-end forces
> from David's RISA model (S0.04 / S9.00). Single-rod fuse concept, telescoping
> columns, and whole-structure overturning basis per call with S. Aher
> (2026-07-30); geometry per the 2026-07-31 IFP set (S3.03 / S3.04 / S9.00).

In [1]:
%%capture

# --- Environment setup -------------------------------------------------
!pip install handcalcs forallpeople numpy pandas

import sys, os
sys.path.append(os.path.abspath('..'))     # project root -> makes `helpers` importable

# One import brings: units (kip, inch, ksi, ...), math (sin, cos, pi, ...),
# sig(), export_notebook(), setup_formatting(), and the handcalcs %%render magic.
from helpers.formatting import *
setup_formatting(3)                          # 3 significant figures + kip-inch unit rendering

## 1  Seismic Design Parameters (ASCE 7-22 / 2025 CBC)

Design spectral values from the Concept BOD (Site Class D, SDC D). `SDS` governs
the short-period plateau; `Ω0`, `Ie` per Table 15.4-1 for a Steel OCBF designed as
a nonbuilding structure similar to a building. `SDS = 1.307 g` gives the locked
`Cs = 0.402`.

In [2]:
%%render params
S_S     = 1.960          # g, mapped $MCE_R$ (Ch. 22)
S_1     = 0.680          # g, mapped $MCE_R$
S_DS    = 1.307          # g, design short-period (governs)
S_D1    = 1.133          # g, design 1-second
R       = 3.250          # Steel OCBF, Table 15.4-1
Omega_0 = 2.000          # overstrength factor
I_e     = 1.000          # Risk Category II
T_a     = 0.182          # s, approx. period (Eq. 12.8-7)
W       = 230*kip        # whole-HAC seismic weight, RISA dead (bulk in interior 5 bays)
h       = 19.48*ft       # top of steel ~19 ft 5-3/4 in (S3.04/1)

<IPython.core.display.Latex object>

## 2  Whole-Structure Base Shear (ELF, ASCE 7-22 §12.8)

Per S. Aher: take the whole-HAC weight and the locked `Cs`. `V` is the reduced
(design) base shear; `Ve = V·(R/Ie) = SDS·W` is the *elastic* demand the system
would see with no ductility — the gap between the two is what the fuse's ductility
has to absorb.

In [3]:
%%render
C_s = S_DS / (R / I_e)          # Eq. 12.8-2  (locked, 0.402)
V   = C_s * W                   # Eq. 12.8-1  whole-HAC base shear
V_e = V * (R / I_e)             # elastic (unreduced) base shear = SDS*W

<IPython.core.display.Latex object>

`V ≈ 92.5 kip` (either principal direction; S. Aher's ~100–120 kip ballpark),
`Ve ≈ 301 kip`. With `Ta = 0.182 s ≪ Ts = 0.87 s` the HAC sits on the short-period
plateau, so it draws near-peak spectral acceleration — the physical root of the
server-acceleration problem the fuse addresses.

## 3  Vertical Seismic Load Effect

Reduces the gravity available for restoring (worst case for uplift).

In [4]:
%%render
E_v = 0.2 * S_DS          # Eq. 12.4-4a  vertical seismic (x D)

<IPython.core.display.Latex object>

## 4  Overturning Demand (equivalent lateral at 0.7 h, per S. Aher)

Apply the whole-HAC shear `V` as a single resultant at `0.7 h` (the simplification
S. Aher gave; use full `h` for a conservative bound, or David's RISA level loads
for the exact distribution).

In [5]:
%%render params
k_h = 0.700              # equivalent-force height factor (0.7h; use 1.0 for full height)

<IPython.core.display.Latex object>

In [6]:
%%render
h_eff = k_h * h                 # equivalent-lateral-force height
M_ot  = V * h_eff               # overturning moment (whole HAC, one direction)

<IPython.core.display.Latex object>

`M_ot ≈ 1,260 kip·ft` at `0.7 h` (≈ 1,800 kip·ft at full height). This is the
demand balanced against self-weight restoring next.

## 5  Transverse Rocking — Which Base Width Restores?

**The bent rotates about its outermost compression-bearing point.** All four base
points telescope (bear in compression) and carry fuses (resist tension), so the
extreme fibers are the **diagonal brace feet at ±7.25 ft (14.5 ft apart)**, not
the vertical columns at ±4 ft (8 ft apart). Push the bent transversely: the
leeward brace foot is the compression toe; the windward brace-foot fuse takes the
tension. Taking moments about the leeward toe, self-weight acts at the centroid
(mid-width) → **restoring lever = 7.25 ft = half of 14.5 ft**. The 8 ft dimension
only sets the interior column-fuse lever, not the restoring lever.

Base-point layout (leeward toe → windward), all levers about the leeward toe:
brace foot `0` → column `3.25 ft` → column `11.25 ft` → brace foot `14.5 ft`.

In [19]:
%%render params
B_feet  = 14.5*ft        # outer brace-foot spread (3'-3 | 8'-0 | 3'-3, S3.04/1) -> restoring width
B_col   = 8.0*ft         # vertical column-to-column spread (interior column-fuse line)
lever_W = 7.25*ft        # self-weight restoring lever = $B_{feet} / 2$
d_bf    = 14.5*ft        # windward brace-foot fuse lever from leeward toe
d_col   = 11.25*ft       # windward column fuse lever from leeward toe

<IPython.core.display.Latex object>

In [8]:
%%render
M_rest_D  = W * lever_W                 # restoring at full dead load (1.0D mechanism)
SR        = M_rest_D / M_ot             # global stability ratio (>1 -> no net rigid-body uplift)
W_up      = (0.9 - E_v) * W             # gravity in the uplift combo (0.9D - Ev)
M_rest_up = W_up * lever_W              # restoring in the uplift combo
net_ot    = M_ot - M_rest_up            # net overturning held by the tension-side fuse
T_bf      = net_ot / d_bf               # transverse brace-foot fuse tension (14.5 ft lever)

<IPython.core.display.Latex object>

**Answer: use the 14.5 ft brace-foot spread (7.25 ft restoring lever), not 8 ft.**
The choice flips the result: with full dead load `SR ≈ 1.32 > 1`, so the bent is
globally stable and barely nets uplift. Had 8 ft (4 ft lever) been used,
`M_rest ≈ 920 kip·ft < M_ot` and you'd predict large false uplift.

Including vertical seismic (`0.9D − Ev` uplift combo): net overturning `≈ 196
kip·ft` → **`T_bf ≈ 13.5 kip`** at the windward brace-foot fuse. On a `1.0D − Ev`
mechanism basis it drops to ≈ 2 kip (≈ 0); with `ρ = 1.3` on the seismic effect it
rises to ≈ 40 kip. Either way the **transverse fuse is small** — the governing
fuse force is in the longitudinal narrow-braced-bay direction (§§6–7). The one
assumption everything rides on: the leeward brace foot must reliably bear through
its telescope; if not, the pivot migrates inboard toward the 8 ft column line and
the restoring lever shrinks — confirm with S. Aher.

## 6  Longitudinal System — Combined Chevron + Moment Frames

The longitudinal line is a **stiffness hybrid**: one concentrated chevron-braced
bay (≈ 2nd interior bay) carries the primary lateral resistance; the other bays
are welded HSS moment frames. Continuous HSS6×6 columns and HSS8×6 beams tie every
bay to a common drift, so the two systems act **in parallel**.

- **Elastic split is stiffness-driven, and the brace wins.** A braced bay resists
  through brace *axial* stiffness (very stiff); a moment frame through beam/column
  *flexure* (soft, plus HSS joint flexibility). The braced bay draws the large
  majority of story shear; the combined moment frames a meaningful minority. Exact
  split is a RISA output — don't assume 95/5.
- **No slab → the beams are the collectors.** Longitudinal inertia is generated
  all along the interior five bays but only the braced bay reaches ground. With no
  diaphragm, the level beams are the drag struts that collect inertia bay-by-bay
  into the brace, carrying **axial collector force** (accumulating toward the
  brace) on top of frame bending.
- **Overturning concentrates at the braced bay → the fuse.** One narrow bay
  resisting most of `V` resolves the overturning as a tension/compression couple
  in that bay's two columns — a large base uplift right there (the fuse location,
  and why the longitudinal fuse force is high, §7). The moment frames shed little
  overturning.
- **Chevron unbalanced load, capped by the fuse.** If the diagonals go inelastic
  (compression to ≈ 0.3Pn, tension to RyFy) a large unbalanced vertical load hits
  the apex beam (AISC 341 §F1). With a yielding base fuse, brace forces are capped
  by the fuse before that develops → the chevron-beam demand is limited to what
  the fuse can deliver. A genuine simplification for the rocking scheme.
- **Torsion — check the plan.** An off-center braced bay (2nd interior) offsets
  the center of longitudinal rigidity from the mass centroid → plan torsion under
  longitudinal shear, resisted (unevenly) by the far moment frames. Confirm the
  braced-bay location on both longitudinal lines; consider a symmetric second
  braced bay to center rigidity (Casa Adelante torsion lesson).
- **Post-yield hand-off.** Once the base fuse yields / the base rocks, the braced
  bay plateaus and the moment frames pick up a larger share, adding backup
  stiffness, redundancy (`ρ`), and re-centering — they are the secondary system,
  not decorative.

Not a formal ASCE 7 dual system unless the moment frames are detailed for ≥ 25% of
`V`; for the OCBF (`R = 3.25`) treat it as braced frame (primary) + moment frames
(secondary / torsional restraint), and model both.

In [9]:
%%render params
B_bay  = 9.667*ft        # central braced bay width (9'-8", S3.03) -> overturning lever
P_colD = 13.5*kip        # gravity pin-down at the uplift column (dead, per BOD)

<IPython.core.display.Latex object>

In [10]:
%%render
T_hand = M_ot / B_bay - P_colD          # single-bay overturning bracket (all V to one bay, no MF sharing)

<IPython.core.display.Latex object>

Crude single-bay bracket `T_hand ≈ 117 kip` (conservative — assigns all of `V`
to one bay and ignores moment-frame sharing). David's RISA base-end forces (§7)
give the design value: longitudinal chevron `T = 93 kip` — lower, because the
moment frames and the real distribution shave it. The hand number brackets RISA,
so the model is sane.

## 7  Governing Base-End Forces (RISA) & Single-Rod Fuse Sizing

Base-end member forces from David's RISA model for the two base details on S9.00
(Detail 1 = short/transverse brace; Detail 5 = long/longitudinal chevron, governs).
Sign `+C / −T`; local **x = axial**, **y = lateral (transverse)**, **z =
longitudinal**. The fuse is a **single ductile threaded rod** inline with the
brace, tying the HSS to a post-installed anchor base plate and acting as a pure
axial element sized to the base-end tension; the base plate, `¾″⌀ Gr. 105`
anchors, gusset, and reinforced pier (S9.00/1, /5) are capacity-protected.

In [11]:
%%render params
# axial +C / -T  |  local x = axial, y = lateral (transverse), z = longitudinal
P_lat_C =  57.189*kip    # transverse brace - compression (Detail 1)
P_lat_T = -38.187*kip    # transverse brace - tension     (Detail 1)
P_lon_C =  102.496*kip   # longitudinal chevron - compression (Detail 5, governs)
P_lon_T = -93.037*kip    # longitudinal chevron - tension     (Detail 5, governs)
P_col_C =  25.886*kip    # braced-bay column - compression
P_col_T = -15.222*kip    # braced-bay column - tension
M_zz    =  4*kip*ft      # z-z connection moment; all other V, M ~ 0

<IPython.core.display.Latex object>

In [23]:
%%render params
T_u = 93.037 * kip     # governing base-end tension ($|P_{lon\_T}|$, chevron - Detail 5)
T_lat = 38.187 * kip   # transverse-brace tension ($|P_{lat\_T}|$ - Detail 1)
F_yr = 105 * ksi       # fuse-rod yield (F1554 Gr. 105, per S9.00)
phi_t = 0.900          # tension-yield resistance factor
d_arm = 9 * inch       # anchor-row lever arm for $M_{zz}$

<IPython.core.display.Latex object>

In [13]:
%%render
T_M   = M_zz / d_arm                  # extra rod tension from the z-z moment couple
T_rod = T_u + T_M                     # governing fuse-rod tension (longitudinal)
A_req = T_rod / (phi_t * F_yr)        # required reduced-shank area
D_req = sqrt(4 * A_req / pi)          # required reduced-shank diameter
A_lat = T_lat / (phi_t * F_yr)        # transverse-brace fuse area
D_lat = sqrt(4 * A_lat / pi)          # transverse-brace fuse diameter
A_bf  = T_bf / (phi_t * F_yr)         # transverse rocking-foot fuse area (from Sec. 5)
D_bf  = sqrt(4 * A_bf / pi)           # transverse rocking-foot fuse diameter

<IPython.core.display.Latex object>

Governing fuse tension `T_rod ≈ 98 kip` (chevron `93` + `Mzz` share `≈ 5`) →
`D_req ≈ 1.15 in` → **1¼″⌀ reduced shank** (`φFy·A ≈ 116 kip ≥ 98`). Transverse
brace `T = 38 kip` → **¾″⌀** (`D_lat ≈ 0.72 in`); transverse rocking-foot fuse
`T_bf ≈ 13.5 kip` → **≈ ½″⌀**.

Quick sizing guide, `A = T / (φFy)`, `φ = 0.9`:

| T (kip) | Gr. 105 | F1554 Gr. 55 | A36 |
|---|---|---|---|
| 50 | ~⅞″ | ~1⅛″ | ~1½″ |
| 100 | ~1¼″ | ~1⅝″ | ~2″ |

**Grade / ductility.** Since the fuse is a discrete reduced rod between couplers,
prefer a **ductile grade for the fuse segment** (good elongation) with
high-strength larger rods / couplers; the coupler-to-coupler length is the yield
length (`Δy = FyL/E`), sized to the rocking uplift. **Un-amplified caveat:** if the
RISA forces already carry `Ω0`, size the ductile fuse to yield at the un-amplified
demand (`÷Ω0 → ≈ 49 kip → ⅞″⌀`) and let only the protected hardware carry the full
values — confirm the governing combination with David. Full base-plate / ACI
318-19 Ch. 17 anchorage and pier design are the Week-2 50% package.

## 8  Results Summary

In [14]:
import pandas as pd
def kv(x, u="kip"):
    return f"{sig(float(mag(x)))} {u}".strip()
ct  = lambda c, t: f"{sig(mag(c))} / {sig(abs(mag(t)))} kip"
mft = lambda x: f"{sig(mag(x)/12.0)} kip\u00b7ft"   # moment in kip-ft (mag() = kip-inch)

rows = [
    ("Whole-HAC base shear  V",              kv(V),                "Cs*W, R = 3.25"),
    ("Elastic base shear  Ve",               kv(V_e),              "V*(R/Ie) = SDS*W"),
    ("Overturning  M_ot  (0.7h)",            mft(M_ot),            "equiv. lateral at 0.7h"),
    ("Global stability  SR = Mrest/Mot",     f"{sig(float(mag(SR)))}\u00d7", "full D -> > 1 stable"),
    ("Transverse restoring lever",           "7.25 ft",            "= 14.5 ft brace-foot spread / 2"),
    ("Transverse rocking fuse  T_bf",        kv(T_bf),             "0.9D - Ev; 14.5 ft lever"),
    ("Long chevron  C / T  (governs)",       ct(P_lon_C, P_lon_T), "longitudinal (RISA)"),
    ("Transverse brace  C / T",              ct(P_lat_C, P_lat_T), "transverse (RISA)"),
    ("Braced-bay column  C / T",             ct(P_col_C, P_col_T), ""),
    ("Long single-bay bracket  T_hand",      kv(T_hand),           "hand; RISA governs"),
    ("Governing fuse tension  T_rod",        kv(T_rod),            "chevron T + Mzz"),
    ("Fuse reduced shank  D_req",            kv(D_req, "in"),      "-> select 1-1/4 in dia"),
    ("Transverse-brace fuse  D_lat",         kv(D_lat, "in"),      "-> 3/4 in dia"),
]
df = pd.DataFrame(rows, columns=["Quantity", "Value", "Note"])
df

,Quantity,Value,Note
0,Whole-HAC base shear V,92.5 kip,"Cs*W, R = 3.25"
1,Elastic base shear Ve,301 kip,V*(R/Ie) = SDS*W
2,Overturning M_ot (0.7h),1260 kip·ft,equiv. lateral at 0.7h
3,Global stability SR = Mrest/Mot,1.32×,full D -> > 1 stable
4,Transverse restoring lever,7.25 ft,= 14.5 ft brace-foot spread / 2
5,Transverse rocking fuse T_bf,13.5 kip,0.9D - Ev; 14.5 ft lever
6,Long chevron C / T (governs),102 / 93 kip,longitudinal (RISA)
7,Transverse brace C / T,57.2 / 38.2 kip,transverse (RISA)
8,Braced-bay column C / T,25.9 / 15.2 kip,
9,Long single-bay bracket T_hand,117 kip,hand; RISA governs


## 9  Reconciliation Targets for the 2D Perform-3D Model

1. **Elastic period `T1 ≈ 0.18 s`** (modal check).
2. **First yield in the rod fuse** at the base-end tension (longitudinal chevron
   `≈ 93 kip`; transverse brace `≈ 38 kip`); the telescoping shim interface
   transfers shear only and permits the small rocking uplift/rotation.
3. **Longitudinal split** — the braced bay draws the majority of `V`, the beams
   carry the collector drag, and the moment frames pick up share after the fuse
   yields.
4. **Pushover plateaus flat** (elastic–plastic axial fuse); force ceiling ≤ fuse
   overstrength (`Ry·Rt·A·Fy`) so base plate, anchors, gusset, and pier stay
   elastic (capacity-design hierarchy).
5. **Global overturning stable** (`SR ≈ 1.3`, full D) — no rigid-body tipping;
   transverse net uplift small (`T_bf ≈ 13.5 kip`).
6. **Reactions reconcile with RISA** at the base ends (§7) and with ACI 318-19
   Ch. 17 anchorage in the 50% package.

## 10  References & Export

**References:** ASCE 7-22 §12.8, §12.4, Table 15.4-1; AISC 360-22 §E3; AISC 341-22
§F1 (OCBF, chevron unbalanced load); ACI 318-19 Ch. 17. Base-end member forces per
David's RISA model; base details **S9.00/1** (short/transverse brace) and
**S9.00/5** (long/longitudinal chevron); longitudinal framing per **S3.03**,
transverse geometry per **S3.04/1** (2026-07-31 IFP set); single-rod fuse concept,
telescoping columns, and whole-structure overturning basis per call with S. Aher
(2026-07-30). Companion to *OCBF_BaseShear_CapacityReconciliation*. Prepared with
`handcalcs` + `forallpeople`.

Export via the helper (writes to the project `outputs/` folder):

In [15]:
# export_notebook("html")   # or "pdf" (WeasyPrint backend)
export_notebook("html")

Detected active notebook: 'Richmond_HAC_LateralSystem_RockingOverturning_FuseForce_v01.ipynb'
Exporting to HTML in: C:\Users\jeffreyda\hand_calcs\outputs
Success! Your file is ready in the 'outputs' folder.
